# Trending YouTube Dataset —(2025–26)
* Hanieh Mirzad - Mahsa Bahri

# 2025-26 Project

You have to work on the Trending YouTube dataset.

### Notes

1. It is mandatory to use GitHub for developing the project.
2. The project must be a jupyter notebook.
3. There is no restriction on the libraries that can be used, nor on the Python version.
4. All questions on the project must be asked in the Discussion forum on the course website.
5. At most 3 students can be in each group. You must create the groups by yourself. You can use the Discussion forum to create the groups.
6. You do not have to send me the project before the discussion.
7. You do not have to prepare any slides for the discussion.
8. You can use AI tools, but you have to describe in the notebook how you have used such tools and you have to show that you have fully understood everything that you have in your project.

### 1. Create a single dataframe with the concatenation of all input csv files, adding a column called `country`

### 2. Extract all videos that have no tag.

### 3. For each channel, determine the total number of views

### 4. Save all rows with disabled comments and disabled ratings, or that have `video_error_or_removed` in a new dataframe called `excluded`, and remove those rows from the original dataframe.

### 5. Add a `like_ratio` column storing the ratio between the number of likes and of dislikes

### 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

### 7. For each interval, determine the number of videos, average number of likes and of dislikes.

### 8. For each tag, determine the number of videos

Notice that `tags` contains a string with several tags.

### 9. Find the tags with the largest number of videos

### 10. For each (`tag`, `country`) pair, compute average ratio likes/dislikes

### 11. For each (`trending_date`, `country`) pair, the video with the largest number of views

### 12. Divide `trending_date` into three columns: `year`, `month`, `day`

### 13. For each (`month`, `country`) pair, the video with the largest number of views

### 14. Read all json files with the video categories

### 15. For each country, determine how many videos have a category that is not assignable.
Note: In your dataset, the video tables are `*.csv.zst` (Zstandard-compressed CSV). We read them directly.


## 0) Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import io


## 1) Path setup and file discovery


In [2]:
#BASE_DIR = Path.home() / 'Desktop' / 'youtube_project'
from pathlib import Path

DATA_DIR = Path('data')

csv_zst_files = sorted(DATA_DIR.glob('*videos.csv.zst'))
json_files = sorted(DATA_DIR.glob('*_category_id.json'))

print('DATA_DIR:', DATA_DIR)
print('Video files (*.csv.zst):', len(csv_zst_files))
print('Category files (*.json):', len(json_files))
csv_zst_files[:5], json_files[:5]


DATA_DIR: data
Video files (*.csv.zst): 10
Category files (*.json): 10


([WindowsPath('data/CAvideos.csv.zst'),
  WindowsPath('data/DEvideos.csv.zst'),
  WindowsPath('data/FRvideos.csv.zst'),
  WindowsPath('data/GBvideos.csv.zst'),
  WindowsPath('data/INvideos.csv.zst')],
 [WindowsPath('data/CA_category_id.json'),
  WindowsPath('data/DE_category_id.json'),
  WindowsPath('data/FR_category_id.json'),
  WindowsPath('data/GB_category_id.json'),
  WindowsPath('data/IN_category_id.json')])

## 2) Read all `*.csv.zst` files and build ONE dataframe + `country` column
We use the first two letters of the filename as the country code (e.g., `CAvideos...` → `CA`).

### Install dependency (one-time)


In [3]:
!pip -q install zstandard

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\commands\install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\metadata\pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 56, in parse
    return Version(version)
  File "C:\Users\mahsa\anaconda3\lib\site-packages

In [4]:
 pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\cli\req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\commands\install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_internal\metadata\pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "C:\Users\mahsa\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 56, in parse
    return Version(version)
  File "C:\Users\mahsa\anaconda3\lib\site-packages

### Read compressed CSV (.zst)

In [5]:
import zstandard as zstd
import io
import pandas as pd
from pathlib import Path

def read_zst_csv(path: Path) -> pd.DataFrame:
    dctx = zstd.ZstdDecompressor()
    with open(path, 'rb') as f, dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(
            reader,
            encoding='utf-8',
            errors='ignore'
        )
        return pd.read_csv(text_stream)

Quick look:

In [6]:
dfs = []

for f in csv_zst_files:
    country = f.name[:2]
    tmp = read_zst_csv(f)
    tmp['country'] = country
    dfs.append(tmp)

print("Number of dataframes:", len(dfs))

df = pd.concat(dfs, ignore_index=True)
df.shape

Number of dataframes: 10


(375942, 17)

In [7]:
df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA


In [8]:
df.columns

Index(['video_id', 'trending_date', 'title', 'channel_title', 'category_id',
       'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count',
       'thumbnail_link', 'comments_disabled', 'ratings_disabled',
       'video_error_or_removed', 'description', 'country'],
      dtype='object')

## 3) (2) Extract all videos that have no tag
In this dataset, no-tag videos are marked with `[none]`.

In [9]:
no_tag_df = df[df['tags'].fillna('').str.strip().eq('[none]')].copy()
no_tag_df.shape


(37698, 17)

## 4) (3) For each channel, total number of views

In [10]:
channel_total_views = (
    df.groupby('channel_title', as_index=False)['views']
      .sum()
      .sort_values('views', ascending=False)
)
channel_total_views.head(20)


,channel_title,views
4564,ChildishGambinoVEVO,11016766510
15536,Marvel Entertainment,10430605449
17726,NickyJamTV,9479859505
18466,Ozuna,8623329509
28412,ibighit,8205572221
6689,DrakeVEVO,7637228580
2788,Bad Bunny,7124207494
2101,ArianaGrandeVevo,6202230488
28621,jypentertainment,5802822913
7047,Ed Sheeran,5775405574


## 5) (4) Create `excluded` and remove rows from `df`
Excluded rows are those with:
- `comments_disabled == True` AND `ratings_disabled == True`, OR
- `video_error_or_removed == True`

In [11]:
mask_excluded = (df['comments_disabled'] & df['ratings_disabled']) | (df['video_error_or_removed'] == True)

excluded = df[mask_excluded].copy()
df = df[~mask_excluded].copy()

excluded.shape, df.shape


((2620, 17), (373322, 17))

## 6) (5) Add `like_ratio` = likes / dislikes
If dislikes is 0, we store `NaN`.

In [12]:
df['like_ratio'] = np.where(df['dislikes'] > 0, df['likes'] / df['dislikes'], np.nan)
df[['likes','dislikes','like_ratio']].head()


,likes,dislikes,like_ratio
0,787425,43420,18.135076
1,127794,1688,75.707346
2,146035,5339,27.352500
3,132239,1989,66.485168
4,1634130,21082,77.513044


## 7) (6) Cluster publish time into 10-minute intervals
We parse `publish_time` as datetime and floor to 10-minute bins.
We also create a label like `02:20-02:30` (UTC).

In [13]:
df['publish_time_dt'] = pd.to_datetime(df['publish_time'], errors='coerce', utc=True)
df['publish_interval_start'] = df['publish_time_dt'].dt.floor('10min')
df['publish_interval_label'] = (
    df['publish_interval_start'].dt.strftime('%H:%M') + '-' +
    (df['publish_interval_start'] + pd.Timedelta(minutes=10)).dt.strftime('%H:%M')
)
df[['publish_time','publish_interval_label']].head()


,publish_time,publish_interval_label
0,2017-11-10T17:00:03.000Z,17:00-17:10
1,2017-11-13T17:00:00.000Z,17:00-17:10
2,2017-11-12T19:05:24.000Z,19:00-19:10
3,2017-11-12T18:01:41.000Z,18:00-18:10
4,2017-11-09T11:04:14.000Z,11:00-11:10


## 8) (7) For each interval: number of videos, avg likes, avg dislikes

In [14]:
interval_stats = (
    df.groupby('publish_interval_label', as_index=False)
      .agg(
          num_videos=('video_id','count'),
          avg_likes=('likes','mean'),
          avg_dislikes=('dislikes','mean')
      )
      .sort_values('publish_interval_label')
)
interval_stats.head(30)


,publish_interval_label,num_videos,avg_likes,avg_dislikes
0,00:00-00:10,2897,61288.115637,3808.149465
1,00:10-00:20,1509,22748.138502,1449.836315
2,00:20-00:30,1241,21378.280419,1072.344883
3,00:30-00:40,1614,36853.560719,955.890954
4,00:40-00:50,1269,42198.623325,1909.301812
5,00:50-01:00,1322,37474.858548,1582.246596
6,01:00-01:10,2742,15980.817287,734.974471
7,01:10-01:20,1389,105969.572354,5892.362131
8,01:20-01:30,1515,24597.400000,1666.618482
9,01:30-01:40,1667,13511.931614,5765.304739


## 9) (8) Prepare a tag table (explode)
`tags` contains multiple tags separated by `|`. We split and `explode`.

In [15]:
tags_series = df['tags'].fillna('').str.strip()
tags_series = tags_series.replace('[none]', '')
tags_series = tags_series.str.replace('"', '', regex=False)

df_tags = (
    df[['video_id','country']]
      .assign(tag=tags_series.str.split('|'))
      .explode('tag')
)
df_tags['tag'] = df_tags['tag'].fillna('').str.strip()
df_tags = df_tags[df_tags['tag'] != '']
df_tags.head()


,video_id,country,tag
0,n1WpP7iowLc,CA,Eminem
0,n1WpP7iowLc,CA,Walk
0,n1WpP7iowLc,CA,On
0,n1WpP7iowLc,CA,Water
0,n1WpP7iowLc,CA,Aftermath/Shady/Interscope


## 10) (8) For each tag: number of videos

In [16]:
tag_counts = (
    df_tags.groupby('tag', as_index=False)
          .agg(num_videos=('video_id','count'))
          .sort_values('num_videos', ascending=False)
)
tag_counts.head(30)


,tag,num_videos
355156,funny,15039
295442,comedy,12351
11681,2018,11383
466003,news,6363
457956,music,5909
11329,2017,5688
585544,video,5612
383352,humor,5057
559398,television,4170
534706,show,4146


## 11) (9) Tags with the largest number of videos

In [17]:
top_tags = tag_counts.head(20)
top_tags


,tag,num_videos
355156,funny,15039
295442,comedy,12351
11681,2018,11383
466003,news,6363
457956,music,5909
11329,2017,5688
585544,video,5612
383352,humor,5057
559398,television,4170
534706,show,4146


## 12) (10) For each (tag, country): average likes/dislikes ratio
We use the `like_ratio` column computed before.

In [18]:
df_tags_lr = df_tags.merge(
    df[['video_id','country','like_ratio']],
    on=['video_id','country'],
    how='left'
)

tag_country_like_ratio = (
    df_tags_lr.groupby(['tag','country'], as_index=False)
              .agg(avg_like_ratio=('like_ratio','mean'))
              .sort_values('avg_like_ratio', ascending=False)
)
tag_country_like_ratio.head(30)


,tag,country,avg_like_ratio
29225,AT&T AUDIENCE,RU,11688.00
29226,AT&T AUDIENCE Network,RU,11688.00
90084,Direct TV,RU,11688.00
29223,AT&T,RU,11688.00
29411,AUDIENCE Network,RU,11688.00
210694,Originals,RU,11688.00
29360,ATT,RU,11688.00
287766,U-Verse,RU,11688.00
90086,DirectTV,RU,11688.00
29227,AT&T Originals,RU,11688.00


## 13) Parse `trending_date`
The dataset uses format `yy.dd.mm` (example: `17.14.11`).
We convert it to datetime to group easily.

In [19]:
df['trending_date_dt'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m', errors='coerce')
df[['trending_date','trending_date_dt']].head()


,trending_date,trending_date_dt
0,17.14.11,2017-11-14
1,17.14.11,2017-11-14
2,17.14.11,2017-11-14
3,17.14.11,2017-11-14
4,17.14.11,2017-11-14


## 14) (11) For each (trending_date, country): video with max views

In [20]:
idx = df.groupby(['trending_date_dt','country'])['views'].idxmax()
top_by_trending_country = df.loc[idx, ['trending_date_dt','country','video_id','title','views']]
top_by_trending_country = top_by_trending_country.sort_values(['trending_date_dt','country'])
top_by_trending_country.head(30)


,trending_date_dt,country,video_id,title,views
4,2017-11-14,CA,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),33523622
40912,2017-11-14,DE,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),33523622
81895,2017-11-14,FR,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),33523622
122451,2017-11-14,GB,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),33523622
161378,2017-11-14,IN,ePO5M5DE01I,Tiger Zinda Hai | Official Trailer | Salman Kh...,35885754
219394,2017-11-14,KR,n1WpP7iowLc,Eminem - Walk On Water (Audio) ft. Beyoncé,17158579
253980,2017-11-14,MX,JzCsM1vtn78,THE LOGANG MADE HISTORY. LOL. AGAIN.,4477587
294429,2017-11-14,RU,U_4BLnJuR3g,12 лайфхаков с термоклеем для хендмейда,2109573
335063,2017-11-14,US,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),33523622
282,2017-11-15,CA,2Vv-BfVoq4g,Ed Sheeran - Perfect (Official Music Video),39082222


## 15) (12) Split trending_date into year, month, day

In [21]:
df['year'] = df['trending_date_dt'].dt.year
df['month'] = df['trending_date_dt'].dt.month
df['day'] = df['trending_date_dt'].dt.day
df[['trending_date_dt','year','month','day']].head()


,trending_date_dt,year,month,day
0,2017-11-14,2017,11,14
1,2017-11-14,2017,11,14
2,2017-11-14,2017,11,14
3,2017-11-14,2017,11,14
4,2017-11-14,2017,11,14


## 16) (13) For each (month, country): video with max views

In [22]:
idx2 = df.groupby(['month','country'])['views'].idxmax()
top_by_month_country = df.loc[idx2, ['month','country','video_id','title','views']]
top_by_month_country = top_by_month_country.sort_values(['month','country'])
top_by_month_country.head(50)


,month,country,video_id,title,views
11214,1,CA,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,43067983
51930,1,DE,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,37728802
92853,1,FR,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,37728802
135208,1,GB,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,90598955
173405,1,IN,dfnCAmr569k,"Taylor Swift - End Game ft. Ed Sheeran, Future",42019590
228682,1,KR,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,37728802
264769,1,MX,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,31680160
306098,1,RU,dfnCAmr569k,"Taylor Swift - End Game ft. Ed Sheeran, Future",23198594
346578,1,US,LsoLEjrDogU,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,57951412
20055,2,CA,xpVfcZ0ZcFM,Drake - God’s Plan,47362934


## 17) (14) Read all category JSON files
We build a dict for each country: `category_id -> category_title`.

In [23]:
category_maps = {}
for jf in json_files:
    country = jf.name[:2]
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    mapping = {}
    for item in data.get('items', []):
        cid = item.get('id')
        title = item.get('snippet', {}).get('title')
        if cid is not None:
            try:
                mapping[int(cid)] = title
            except:
                pass
    category_maps[country] = mapping

list(category_maps.keys())[:10], len(category_maps)


(['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US'], 10)

## 18) (15) For each country: how many videos have a category that is not assignable
A category is 'not assignable' if its `category_id` is not present in that country's JSON mapping.

In [24]:
def map_category(row):
    mapping = category_maps.get(row['country'], {})
    try:
        return mapping.get(int(row['category_id']), np.nan)
    except:
        return np.nan

df['category_title'] = df.apply(map_category, axis=1)

not_assignable_counts = (
    df.groupby('country', as_index=False)
      .agg(not_assignable=('category_title', lambda s: s.isna().sum()))
      .sort_values('not_assignable', ascending=False)
)
not_assignable_counts


,country,not_assignable
8,RU,1538
6,KR,286
7,MX,251
1,DE,242
2,FR,111
3,GB,90
0,CA,74
4,IN,74
5,JP,18
9,US,0


## How we used AI tools

we used an AI tool to help Us organize the notebook structure and recall basic pandas operations
such as `groupby`, `merge`, and `explode`.

All the code was executed and tested by me on the provided dataset.
I checked the results step by step and I understand every operation used in this notebook.